# Alerts Deepdive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AiMTT-project/use-case-1/blob/main/3.2%20Alerting/Assignment/alerts_deepdive.ipynb)


### Introduction to Alert-Based Monitoring

Detailed information can be highly valuable, as it provides insight and supports situational awareness. However, too much information can quickly become overwhelming and reduce clarity, especially in time-critical environments.

During the preparation for SAIL 2025, the Amsterdam-Amstelland safety region expressed a strong need to move towards **alert-driven workflows**. Instead of continuously monitoring large amounts of data, the goal is to highlight only the most relevant situations. A key ambition within this approach is the development of **predictive alerts**, enabling earlier and more proactive decision-making.

In this notebook, we explore how to design effective alerts based on existing data streams. We focus on:

- Defining alerts using threshold values  
- Identifying meaningful moments that require attention  
- Visualising alerts in a clear and actionable way  

The aim is to transform raw data into **concise, decision-supporting signals** for crowd and event management.

### Import necessary libraries

In [ ]:
!pip install geopandas contextily matplotlib numpy

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from pyproj import Transformer


### Define functions

In [ ]:
# @title
def plot_los_heatmap(
    df,
    time_col="time",
    location_col="location",
    los_col="LoS",
    time_freq="1min",
    location_order=None,
    figsize=(14, 5)
):
    """
    Plot a heatmap of pedestrian Level of Service (LoS) over time per location.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing time, location, and LoS columns.
    time_col : str
        Name of the datetime column.
    location_col : str
        Name of the location column.
    los_col : str
        Name of the LoS column (values like 'LoS A', 'LoS B', ..., 'LoS F').
    time_freq : str
        Resampling frequency for time aggregation, e.g. '10s', '1min', '5min'.
    location_order : list or None
        Optional custom order of locations.
    figsize : tuple
        Figure size.
    """

    data = df.copy()
    data[time_col] = pd.to_datetime(data[time_col])

    # Standardise LoS labels, now including LoS F
    los_mapping = {
        "LoS A": 0,
        "LoS B": 1,
        "LoS C": 2,
        "LoS D": 3,
        "LoS E": 4,
        "LoS F": 5,
        "A": 0,
        "B": 1,
        "C": 2,
        "D": 3,
        "E": 4,
        "F": 5,
    }

    data["los_num"] = data[los_col].map(los_mapping)

    if data["los_num"].isna().any():
        invalid = data.loc[data["los_num"].isna(), los_col].unique()
        raise ValueError(f"Unknown LoS values found: {invalid}")

    # Round timestamps to a common interval
    data["time_bin"] = data[time_col].dt.floor(time_freq)

    # If multiple records exist per location/time bin, take the worst LoS
    grouped = (
        data.groupby([location_col, "time_bin"], as_index=False)["los_num"]
        .max()
    )

    # Pivot to matrix
    heatmap_df = grouped.pivot(
        index=location_col,
        columns="time_bin",
        values="los_num"
    )

    # Reorder locations if needed
    if location_order is not None:
        heatmap_df = heatmap_df.reindex(location_order)

    values = heatmap_df.values

    # Discrete colormap for LoS A-F
    cmap = mcolors.ListedColormap([
        "#4CAF50",  # LoS A - green
        "#CDDC39",  # LoS B
        "#FFEB3B",  # LoS C - yellow
        "#FF9800",  # LoS D - orange
        "#F44336",  # LoS E - red
        "#B71C1C",  # LoS F - darker red
    ])
    bounds = np.arange(-0.5, 6.5, 1)
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(values, aspect="auto", cmap=cmap, norm=norm)

    # Axis labels
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)

    # Reduce number of x tick labels for readability
    time_labels = heatmap_df.columns
    if len(time_labels) > 12:
        tick_idx = np.linspace(0, len(time_labels) - 1, 12, dtype=int)
    else:
        tick_idx = np.arange(len(time_labels))

    ax.set_xticks(tick_idx)
    ax.set_xticklabels(
        [time_labels[i].strftime("%m-%d %H:%M") for i in tick_idx],
        rotation=45,
        ha="right"
    )

    ax.set_xlabel("Time")
    ax.set_ylabel("Location")
    ax.set_title("Pedestrian Level of Service per Location Over Time")

    # Colorbar
    cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4, 5])
    cbar.ax.set_yticklabels(["LoS A", "LoS B", "LoS C", "LoS D", "LoS E", "LoS F"])
    cbar.set_label("Level of Service")

    plt.tight_layout()
    plt.show()

### Retrieve necessary data

This module works with multi-source monitoring datasets:
- `los_alerts_deepdive_data.geojson`: Pedestrian Level of Service data
- `visualisation_deepdive_data.geojson`: Crowdscan pedestrian density data
- `tomtom_alerts_deepdive_data.geojson`: Road traffic congestion data
- `vaarwegen_alerts_deepdive_data.geojson`: Waterway vessel traffic data

The helper functions below automatically resolve these files locally or download them from GitHub into `sample_data/`.


In [ ]:
import os
import urllib.request

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AiMTT-project/use-case-1/main"
GITHUB_LFS_BASE = "https://media.githubusercontent.com/media/AiMTT-project/use-case-1/main"

def resolve_dataset(filename, github_subfolder="3.2%20Alerting"):
    """Resolves dataset from local directory or downloads from GitHub (with Git LFS support)."""
    candidate_paths = [
        filename,
        os.path.join("..", filename),
        os.path.join("sample_data", filename),
        os.path.join("/content/sample_data", filename),
        os.path.join("..", "..", "2.2 LOS", filename),
        os.path.join("..", "..", "2.3 Visualisatie", filename),
        os.path.join("..", "..", "3.2 Alerting", filename),
    ]
    for p in candidate_paths:
        if os.path.exists(p):
            # If local file is larger than 1KB (not an unhydrated LFS pointer), use it
            if os.path.getsize(p) > 1024 or not filename.endswith(".geojson"):
                return p

    os.makedirs("sample_data", exist_ok=True)
    target_path = os.path.join("sample_data", filename)

    # Download if missing or if existing file is an LFS pointer stub
    if not os.path.exists(target_path) or (filename.endswith(".geojson") and os.path.getsize(target_path) < 1024):
        # Git LFS tracked files must be fetched from media.githubusercontent.com
        if "tomtom" in filename or "vaarwegen" in filename:
            url = f"{GITHUB_LFS_BASE}/{github_subfolder}/{filename}"
        else:
            url = f"{GITHUB_RAW_BASE}/{github_subfolder}/{filename}"
            
        print(f"Downloading {filename} from GitHub...")
        try:
            urllib.request.urlretrieve(url, target_path)
            print(f"Saved {filename} ({os.path.getsize(target_path)/(1024*1024):.2f} MB) to {target_path}")
        except Exception as e:
            print(f"Could not auto-download {filename}: {e}")
            
    return target_path

lvma_path = resolve_dataset("los_alerts_deepdive_data.geojson", "2.2%20LOS")
print(f"Using LVMA dataset: {lvma_path}")


In [ ]:
gdf = gpd.read_file(lvma_path)

### Understanding the data

First off, we are going to perform some basic exploration of the data we'll be working with. To start, lets explore which attributes the dataset has.

In [ ]:
# Print all columns in gdf and their data types in a clear table
df_gdf_info = pd.DataFrame({
    "Attribute": gdf.columns,
    "Data Type": [gdf[col].dtype for col in gdf.columns]
})
print(df_gdf_info.to_string(index=False))

In [ ]:
gdf.head(2)

So we are working with a dataset which tracks counts for multiple sensors and locations over a period of time. Lets see what we can find out about the time granularity and extent of the geodataframe.

In [ ]:
# Print start and end time, and time granularity stats
start_time = gdf['time'].min()
end_time = gdf['time'].max()

print(f"Start time: {start_time}")
print(f"End time: {end_time}")
# Show the most common and average time between datapoints in seconds
# Calculate and print the average time between datapoints, grouped by location
gdf['time'] = pd.to_datetime(gdf['time'])
avg_time_by_location = (
    gdf.sort_values(['location', 'time'])
    .groupby('location')['time']
    .apply(lambda x: x.diff().dropna().mean())
)
print("Average time between datapoints by location:")
print(avg_time_by_location)

This shows that we are working with data from 20 August between 08:00 and 16:30. It also indicates that the dataset includes multiple locations with a time granularity of approximately 1.5 to 2 minutes. Let's explore which locations we'll be working with.

In [ ]:

# Plot all areas using their geometry on an OpenStreetMap basemap
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, column='location', legend=True, alpha=0.5, edgecolor='k')
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=gdf.crs)
ax.set_title("All Areas by Location on OpenStreetMap")
plt.axis('off')
plt.show()

So we are exploring multiple areas, mostly bridges around Central Station in Amsterdam on the 20th of August, the first day of SAIL 2025. Finally, let's examine the actual counts the sensored measured on this day.

In [ ]:
plot_los_heatmap(gdf)

The graph highlights a peak in Level of Service (LoS) across several locations between 17:10 and 19:58. This period will be examined in more detail to investigate how alerting mechanisms can support concise and actionable decision-making during events.For more information about LoS, see the material provided earlier on in the learning module this notebook is part of.

### Information overload

Information overload in map visualisations occurs when too much data is shown at once, making maps difficult to read and slowing decision-making.

For information managers, combining many data layers (like weather, incidents, and infrastructure) can clutter the view and hide key insights. For crowd managers, who depend on quick interpretation of crowd density and movement, overloaded maps can obscure urgent risks such as bottlenecks or unsafe crowd build-ups.

This matters because it can delay reactions, increase misinterpretation, and hide critical patterns.

*Assignment 1*: Load in all the datasets. Then run all prewritten code blocks and inspect the visualisation.


In [ ]:
tomtom_path = resolve_dataset("tomtom_alerts_deepdive_data.geojson", "3.2%20Alerting")
crowdscan_path = resolve_dataset("visualisation_deepdive_data.geojson", "2.3%20Visualisatie")
vaarwegen_path = resolve_dataset("vaarwegen_alerts_deepdive_data.geojson", "3.2%20Alerting")

print(f"Using TomTom dataset: {tomtom_path}")
print(f"Using Crowdscan dataset: {crowdscan_path}")
print(f"Using Vaarwegen dataset: {vaarwegen_path}")


In [ ]:
tomtom_gdf = gpd.read_file(tomtom_path)
crowdscan_gdf = gpd.read_file(crowdscan_path)
vaarwegen_gdf = gpd.read_file(vaarwegen_path)
crowdscan_gdf = crowdscan_gdf.to_crs(gdf.crs)
vaarwegen_gdf = vaarwegen_gdf.to_crs(gdf.crs)


In [ ]:
# Target time
target_time = pd.Timestamp("2025-08-21 14:30:00+02:00")
window_start = target_time - pd.Timedelta(minutes=1)
window_end = target_time

# Ensure datetime columns
gdf['time'] = pd.to_datetime(gdf['time'])
vaarwegen_gdf['timestamp'] = pd.to_datetime(vaarwegen_gdf['timestamp'])
crowdscan_gdf['time'] = pd.to_datetime(crowdscan_gdf['time'])
tomtom_gdf['time'] = pd.to_datetime(tomtom_gdf['time'])

# Filter all datasets
gdf_filtered = gdf[
    (gdf['time'] >= window_start) &
    (gdf['time'] <= window_end)
]

vaarwegen_gdf_filtered = vaarwegen_gdf[
    (vaarwegen_gdf['timestamp'] >= window_start) &
    (vaarwegen_gdf['timestamp'] <= window_end)
]

crowdscan_gdf_filtered = crowdscan_gdf[
    (crowdscan_gdf['time'] >= window_start) &
    (crowdscan_gdf['time'] <= window_end)
]

tomtom_gdf_filtered = tomtom_gdf[
    (tomtom_gdf['time'] >= window_start) &
    (tomtom_gdf['time'] <= window_end)
]

# Collect layers
layers = []
for gdf_, name in [
    (gdf_filtered, "Level of Service"),
    (vaarwegen_gdf_filtered, "Vaarwegen"),
    (crowdscan_gdf_filtered, "Crowdscan"),
    (tomtom_gdf_filtered, "TomTom"),
]:
    if not gdf_.empty:
        if gdf_.crs is None:
            raise ValueError(f"{name} has no CRS defined.")
        layers.append((gdf_.to_crs(epsg=3857), name))

# Create figure first
fig, ax = plt.subplots(figsize=(12, 12))

colors = {
    "Level of Service": "purple",
    "Vaarwegen": "blue",
    "Crowdscan": "green",
    "TomTom": "red",
}

# Plot layers
for gdf_, name in layers:
    gdf_.plot(ax=ax, color=colors[name], alpha=0.5)

# Amsterdam city centre + IJ bounding box in lon/lat
# Roughly: west, south, east, north
lon_min, lat_min = 4.86, 52.36
lon_max, lat_max = 5.00, 52.41

# Convert WGS84 -> Web Mercator
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
xmin, ymin = transformer.transform(lon_min, lat_min)
xmax, ymax = transformer.transform(lon_max, lat_max)

# Set extent AFTER creating ax
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Add basemap after setting extent
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

# Legend only for layers that are present
legend_handles = [
    Line2D([0], [0], marker='o', color='w', label=name,
           markerfacecolor=colors[name], markersize=10)
    for _, name in layers
]
if legend_handles:
    ax.legend(handles=legend_handles)

ax.set_title("All Data Sources between 14:29 and 14:30 on 2025-08-21 (UTC+2)")
ax.axis("off")

plt.show()

### Generating alerts

Using notification-based (e.g. alerts) information helps counter information overload by only presenting data when it is relevant or requires action, instead of continuously displaying everything on a map.

For information and crowd managers, this means they can focus on a clean, simple overview while receiving alerts for important changes: such as rising crowd density, unusual movement patterns, or incidents. This reduces distraction and ensures attention is directed to what matters most in the moment.

In practice, notifications can be used for threshold alerts (when a crowd exceeds safe limits), real-time incident warnings, or predictive alerts based on trends or (AI) algorithms. This allows faster responses, better prioritization, and more effective decision-making without overwhelming the user.

*Assignment 2*: Filter all datasets on minute 18.34 on the 21st of August. You can use the provided filter_nearest function. Visualize the data again. However, this time only visualize the data from the Level of Service dataset (gdf) when the LoS is == LoS F. Use the provided icon to visualize this dataset. Visualize the other datasources in the same way as before.

In [ ]:
import os
import urllib.request
from PIL import Image, ImageDraw

# Setup warning icon (compatible locally and in Colab)
icon_filename = "warning.png"
candidate_icon_paths = [
    icon_filename,
    os.path.join("..", icon_filename),
    os.path.join("sample_data", icon_filename),
    os.path.join("/content/sample_data", icon_filename),
    os.path.join("/content", icon_filename),
    os.path.join("..", "..", "3.2 Alerting", icon_filename),
]

warning_icon = None
for p in candidate_icon_paths:
    if os.path.exists(p) and os.path.getsize(p) > 100:
        warning_icon = p
        break

if warning_icon is None:
    os.makedirs("sample_data", exist_ok=True)
    warning_icon = os.path.join("sample_data", icon_filename)
    icon_url = "https://raw.githubusercontent.com/AiMTT-project/use-case-1/main/3.2%20Alerting/warning.png"
    try:
        print(f"Downloading {icon_filename} from GitHub...")
        req = urllib.request.Request(icon_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(warning_icon, "wb") as f:
            f.write(resp.read())
        print(f"Saved to {warning_icon}")
    except Exception as e:
        print(f"Creating fallback warning icon: {e}")
        img = Image.new("RGBA", (128, 128), (0, 0, 0, 0))
        draw = ImageDraw.Draw(img)
        draw.polygon([(64, 10), (10, 118), (118, 118)], fill="#F1C73A", outline="#8A3817", width=6)
        draw.rectangle([(60, 45), (68, 85)], fill="#000000")
        draw.ellipse([(60, 95), (68, 103)], fill="#000000")
        img.save(warning_icon)

print(f"Using warning icon: {warning_icon}")


In [ ]:
target_time = pd.Timestamp("2025-08-21 18:34:00+02:00")

In [ ]:
def filter_nearest(df, time_col, target_time):
    # Compute absolute time difference
    time_diff = (df[time_col] - target_time).abs()

    # Find closest timestamp
    closest_time = df.loc[time_diff.idxmin(), time_col]

    # Return rows within ±30 seconds of closest time
    return df[
        (df[time_col] >= closest_time - pd.Timedelta(seconds=30)) &
        (df[time_col] <= closest_time + pd.Timedelta(seconds=30))
    ]


gdf_time_filtered = filter_nearest(gdf, 'time', target_time)
vaarwegen_gdf_time_filtered = filter_nearest(vaarwegen_gdf, 'timestamp', target_time)
crowdscan_gdf_time_filtered = filter_nearest(crowdscan_gdf, 'time', target_time)
tomtom_gdf_time_filtered = filter_nearest(tomtom_gdf, 'time', target_time)

#### How to Determine Important Signals

- **Collaborate with crowd and information managers**  
  Important signals should always be defined together with both crowd managers and information managers. Crowd managers understand real-world risks and operational needs, while information managers understand the data and technical possibilities. Combining these perspectives ensures signals are both meaningful and actionable.

- **Test and refine before the event**  
  It is important to practice and validate signals in advance using simulations, historical data, or small-scale tests. This helps determine which signals are useful, set appropriate thresholds, and improve how notifications are delivered. It also builds trust and familiarity with the system.

- **Use moving windows and aggregated metrics**  
  Using moving windows (rolling time intervals) with averages or similar metrics helps reduce noise in the data. Instead of reacting to sudden spikes, this approach highlights consistent trends, such as gradually increasing crowd density. This leads to fewer false alarms and more reliable alerts. Other metrics to consider are moving windows with medians or averages in period windows.

*Assignment 3.1*: Apply a moving average to the same Level of Service dataset to determine when an alert should be provided. Apply the window during 18.15 and 18.45.
Apply the average to the flow and density. Only visualize the Level of Service alerts. For you interval, take into account that the datapoints are pushed every two minutes. The average flow and density during the window should be:
-  Density > 3.0
-  Flow < 82

In [ ]:
# Filter gdf for records between 18:15 and 18:45 on 2025-08-21
start = pd.Timestamp("2025-08-21 18:15:00+02:00")
end = pd.Timestamp("2025-08-21 18:45:00+02:00")
gdf_time_window = gdf[(gdf['time'] >= start) & (gdf['time'] <= end)]

In [ ]:
def plot_los_f_alerts_over_time():
    # Step 1: Prepare environment (create output folder, copy data, convert time, flow, and density to correct formats).

    # Step 2: Filter data to the selected time range and create time bins (frame_time).

    # Step 3: Reproject data and extract one representative geometry per location.

    # Step 4: Aggregate flow and density per location and time bin, and create a complete time-location grid.

    # Step 5: Compute moving averages of flow and density per location over the chosen window.

    # Step 6: Apply LoS F thresholds (density ≥ 3.0 and flow ≤ 82) to identify alert moments.

    # Step 7: Loop over each time step to plot alerts with icons on a basemap and save each frame as an image.

    # Step 8: Combine all images into an MP4 animation and save the result.
    pass

*Assignment 3.2*: Apply a moving minimum to the same Level of Service dataset to determine when an alert should be provided. This indicates that all values in a given window should be LoS F. Only visualize the Level of Service alerts. Again save it to an mp4.

In [ ]:
def plot_los_f_alerts_over_time():
    # Step 1: Prepare the data (copy input, convert time, map LoS categories to numeric values, and create the output folder).

    # Step 2: Filter the dataset to the selected time range and assign each row to a time bin.

    # Step 3: Reproject the data and extract one representative geometry per location for plotting.

    # Step 4: Aggregate LoS per location and time bin, then build a complete location-time grid.

    # Step 5: Fill missing bins, sort the data, and compute the rolling minimum LoS per location.

    # Step 6: Select alert moments where the rolling minimum equals LoS F and attach geometry.

    # Step 7: Loop through each time step to plot alert icons on the basemap and save each frame as an image.

    # Step 8: Combine the saved images into an MP4 animation and save the final result.
    pass

### Conclusion

This notebook demonstrates how data scientists can translate theoretical crowd management concepts into real-time analytical tools for operational use. Instead of focusing on raw data, it emphasizes the importance of deriving meaningful indicators, like Level of Service, from flow and density, and applying temporal aggregation (e.g. moving averages) to reflect sustained conditions rather than momentary fluctuations.

The results show that effective real-time insights require balancing sensitivity and stability: instantaneous data is too noisy, while aggregated metrics better support decision-making. By implementing threshold-based alerts and smoothing techniques, the analysis aligns with the theory that crowd management systems should provide clear, actionable information to crowd and information managers during events.